In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("ACN_Preprocessing") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.executor.memory", "3g") \
    .getOrCreate()

print("="*70)
print("STEP 1: LOAD RAW DATA")
print("="*70)

# Đọc toàn bộ dữ liệu thô (giữ nguyên các cột)
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .parquet("hdfs://localhost:9000/ev-project/data/silver/ev_sessions/caltech/*")

print(f"Raw sessions: {df_raw.count():,}")
print(f"Raw columns: {df_raw.columns}")
df_raw.printSchema()

26/04/17 14:05:05 WARN Utils: Your hostname, ax2 resolves to a loopback address: 127.0.1.1; using 192.168.40.130 instead (on interface ens33)
26/04/17 14:05:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 14:05:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


STEP 1: LOAD RAW DATA


26/04/17 14:05:28 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
[Stage 2:>                                                          (0 + 4) / 4]

Raw sessions: 31,424
Raw columns: ['_batch_id', '_id', '_ingest_time', 'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime', 'kWhDelivered', 'sessionID', 'siteID', 'spaceID', 'stationID', 'timezone', 'userID', 'userInputs']
root
 |-- _batch_id: string (nullable = true)
 |-- _id: string (nullable = true)
 |-- _ingest_time: string (nullable = true)
 |-- clusterID: string (nullable = true)
 |-- connectionTime: string (nullable = true)
 |-- disconnectTime: string (nullable = true)
 |-- doneChargingTime: string (nullable = true)
 |-- kWhDelivered: double (nullable = true)
 |-- sessionID: string (nullable = true)
 |-- siteID: string (nullable = true)
 |-- spaceID: string (nullable = true)
 |-- stationID: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- userID: string (nullable = true)
 |-- userInputs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- WhPerMile: long (nullable = true)
 |    |    |-- kWhRequested: double 

In [2]:
print("="*70)
print("STEP 2: REMOVE ID COLUMNS")
print("="*70)

# Paper: "_id, sessionID, stationID, spaceID, siteID, clusterID, userID"
id_columns = ['_batch_id', '_id', '_ingest_time', 'sessionID', 
                        'stationID', 'spaceID', 'siteID', 'clusterID', 'userID']

# Kiểm tra xem có tồn tại không
existing_cols = [c for c in id_columns if c in df_raw.columns]
missing_cols = [c for c in id_columns if c not in df_raw.columns]

print(f"Removing: {existing_cols}")
if missing_cols:
    print(f"⚠️ Missing (already not in data): {missing_cols}")
    
df_step2 = df_raw.drop(*id_columns)

print(f"Removed columns: {id_columns}")
print(f"Remaining columns: {df_step2.columns}")
print(f"Row count: {df_step2.count():,}")
df_step2.printSchema()

STEP 2: REMOVE ID COLUMNS
Removing: ['_batch_id', '_id', '_ingest_time', 'sessionID', 'stationID', 'spaceID', 'siteID', 'clusterID', 'userID']
Removed columns: ['_batch_id', '_id', '_ingest_time', 'sessionID', 'stationID', 'spaceID', 'siteID', 'clusterID', 'userID']
Remaining columns: ['connectionTime', 'disconnectTime', 'doneChargingTime', 'kWhDelivered', 'timezone', 'userInputs']


[Stage 5:==============>                                            (1 + 3) / 4]

Row count: 31,424
root
 |-- connectionTime: string (nullable = true)
 |-- disconnectTime: string (nullable = true)
 |-- doneChargingTime: string (nullable = true)
 |-- kWhDelivered: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- userInputs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- WhPerMile: long (nullable = true)
 |    |    |-- kWhRequested: double (nullable = true)
 |    |    |-- milesRequested: long (nullable = true)
 |    |    |-- minutesAvailable: long (nullable = true)
 |    |    |-- modifiedAt: string (nullable = true)
 |    |    |-- paymentRequired: boolean (nullable = true)
 |    |    |-- requestedDeparture: string (nullable = true)
 |    |    |-- userID: long (nullable = true)



In [3]:
print("="*70)
print("STEP 3: CONVERT DATETIME TO TIMEZONE-NAIVE")
print("="*70)

# Kiểm tra format datetime hiện tại
print("📋 Raw datetime samples:")
df_step2.select("connectionTime", "disconnectTime", "doneChargingTime", "timezone").show(5, truncate=False)

# Paper: convert to timezone-naive format
# Dữ liệu có dạng: "2018-09-25 22:42:06-07:00"
# Dùng from_utc_timestamp để chuyển về UTC và bỏ timezone

df_step3 = df_step2 \
    .withColumn("conn_ts_with_tz", 
                to_timestamp(col("connectionTime"), "yyyy-MM-dd HH:mm:ssXXX")) \
    .withColumn("disc_ts_with_tz", 
                to_timestamp(col("disconnectTime"), "yyyy-MM-dd HH:mm:ssXXX")) \
    .withColumn("done_ts_with_tz", 
                to_timestamp(col("doneChargingTime"), "yyyy-MM-dd HH:mm:ssXXX"))

# Chuyển về UTC (bỏ timezone)
df_step3 = df_step3 \
    .withColumn("connectionTime_utc", 
                from_utc_timestamp(col("conn_ts_with_tz"), "UTC")) \
    .withColumn("disconnectTime_utc", 
                from_utc_timestamp(col("disc_ts_with_tz"), "UTC")) \
    .withColumn("doneChargingTime_utc", 
                from_utc_timestamp(col("done_ts_with_tz"), "UTC")) \
    .drop("connectionTime", "disconnectTime", "doneChargingTime", "timezone",
          "conn_ts_with_tz", "disc_ts_with_tz", "done_ts_with_tz")

print("\n✅ After conversion (timezone-naive UTC):")
df_step3.select("connectionTime_utc", "disconnectTime_utc", "doneChargingTime_utc").show(5, truncate=False)

# Kiểm tra null counts
null_counts = df_step3.select(
    sum(col("connectionTime_utc").isNull().cast("int")).alias("conn_null"),
    sum(col("disconnectTime_utc").isNull().cast("int")).alias("disc_null"),
    sum(col("doneChargingTime_utc").isNull().cast("int")).alias("done_null")
).collect()[0]

print(f"\n📊 Null counts after conversion:")
print(f"  connectionTime_utc null: {null_counts['conn_null']:,}")
print(f"  disconnectTime_utc null: {null_counts['disc_null']:,}")
print(f"  doneChargingTime_utc null: {null_counts['done_null']:,}")

# Kiểm tra time range
print("\n📅 Time range:")
df_step3.select(
    min("connectionTime_utc").alias("earliest"),
    max("connectionTime_utc").alias("latest")
).show()

print(f"\n✅ Remaining columns: {df_step3.columns}")
print(f"✅ Row count: {df_step3.count():,}")

# Bỏ userInputs (paper không dùng)
df_step3 = df_step3.drop("userInputs")
print(f"\n✅ Dropped 'userInputs' (not used in paper)")
print(f"✅ Final columns for next step: {df_step3.columns}")

STEP 3: CONVERT DATETIME TO TIMEZONE-NAIVE
📋 Raw datetime samples:


+-------------------------+-------------------------+-------------------------+-------------------+
|connectionTime           |disconnectTime           |doneChargingTime         |timezone           |
+-------------------------+-------------------------+-------------------------+-------------------+
|2018-09-25 22:42:06-07:00|2018-09-25 22:54:07-07:00|2018-09-25 22:54:03-07:00|America/Los_Angeles|
|2018-09-25 22:03:35-07:00|2018-09-25 22:18:53-07:00|2018-09-25 22:18:49-07:00|America/Los_Angeles|
|2018-09-25 21:35:26-07:00|2018-09-26 07:33:41-07:00|2018-09-26 07:01:30-07:00|America/Los_Angeles|
|2018-09-25 21:34:24-07:00|2018-09-25 22:26:26-07:00|2018-09-25 22:21:37-07:00|America/Los_Angeles|
|2018-09-25 20:36:28-07:00|2018-09-26 09:43:48-07:00|2018-09-25 22:45:03-07:00|America/Los_Angeles|
+-------------------------+-------------------------+-------------------------+-------------------+
only showing top 5 rows


✅ After conversion (timezone-naive UTC):
+-------------------+------------


📊 Null counts after conversion:
  connectionTime_utc null: 0
  disconnectTime_utc null: 0
  doneChargingTime_utc null: 2,055

📅 Time range:


+-------------------+-------------------+
|           earliest|             latest|
+-------------------+-------------------+
|2018-04-25 11:08:04|2021-09-14 01:52:37|
+-------------------+-------------------+


✅ Remaining columns: ['kWhDelivered', 'userInputs', 'connectionTime_utc', 'disconnectTime_utc', 'doneChargingTime_utc']
✅ Row count: 31,424

✅ Dropped 'userInputs' (not used in paper)
✅ Final columns for next step: ['kWhDelivered', 'connectionTime_utc', 'disconnectTime_utc', 'doneChargingTime_utc']


In [4]:
print("="*70)
print("STEP 4: DURATION FEATURES (in hours)")
print("="*70)

from pyspark.sql.functions import unix_timestamp, col, log1p, when

# Paper: duration = disconnectTime - connectionTime (hours)
# Paper: charging_duration = doneChargingTime - connectionTime (hours)

df_step4 = df_step3 \
    .withColumn("duration", 
                (unix_timestamp(col("disconnectTime_utc")) - 
                 unix_timestamp(col("connectionTime_utc"))) / 3600) \
    .withColumn("charging_duration", 
                when(col("doneChargingTime_utc").isNotNull(),
                     (unix_timestamp(col("doneChargingTime_utc")) - 
                      unix_timestamp(col("connectionTime_utc"))) / 3600)
                .otherwise(None))

# Log transform of charging_duration (paper: charging_duration_log)
df_step4 = df_step4 \
    .withColumn("charging_duration_log", 
                when(col("charging_duration") > 0, log1p(col("charging_duration")))
                .otherwise(None))

print("📊 Duration statistics (hours):")
df_step4.select("duration", "charging_duration", "charging_duration_log").describe().show()

print("\n📋 Sample duration features (first 10 rows):")
df_step4.select(
    "connectionTime_utc", "disconnectTime_utc", "doneChargingTime_utc",
    "duration", "charging_duration", "charging_duration_log"
).show(10, truncate=False)

# Kiểm tra giá trị không hợp lệ
print("\n📊 Data quality checks:")
invalid_duration = df_step4.filter(col("duration") < 0).count()
invalid_charging = df_step4.filter(col("charging_duration") < 0).count()
print(f"  Negative duration: {invalid_duration:,}")
print(f"  Negative charging_duration: {invalid_charging:,}")

# Kiểm tra phân bố
print("\n📊 Duration distribution (hours):")
df_step4.select("duration").summary("min", "25%", "50%", "75%", "max").show()

print(f"\n✅ Remaining columns: {df_step4.columns}")
print(f"✅ Row count: {df_step4.count():,}")

STEP 4: DURATION FEATURES (in hours)
📊 Duration statistics (hours):


+-------+--------------------+-------------------+---------------------+
|summary|            duration|  charging_duration|charging_duration_log|
+-------+--------------------+-------------------+---------------------+
|  count|               31424|              29369|                29341|
|   mean|   5.652122086444909| 2.9775439333234828|   1.1978547438016065|
| stddev|   5.963686144541357| 3.4296022505991206|   0.5716383654504634|
|    min|0.034444444444444444|-0.6894444444444444| 8.329863038918628E-4|
|    max|  245.26916666666668| 200.01583333333335|    5.303383677759315|
+-------+--------------------+-------------------+---------------------+


📋 Sample duration features (first 10 rows):
+-------------------+-------------------+--------------------+-------------------+-------------------+---------------------+
|connectionTime_utc |disconnectTime_utc |doneChargingTime_utc|duration           |charging_duration  |charging_duration_log|
+-------------------+-------------------+------

  Negative duration: 0
  Negative charging_duration: 26

📊 Duration distribution (hours):


+-------+--------------------+
|summary|            duration|
+-------+--------------------+
|    min|0.034444444444444444|
|    25%|  1.8663888888888889|
|    50%|   4.729722222222223|
|    75%|   8.465833333333334|
|    max|  245.26916666666668|
+-------+--------------------+


✅ Remaining columns: ['kWhDelivered', 'connectionTime_utc', 'disconnectTime_utc', 'doneChargingTime_utc', 'duration', 'charging_duration', 'charging_duration_log']
✅ Row count: 31,424


In [5]:
print("="*70)
print("STEP 5: TEMPORAL FEATURES")
print("="*70)

from pyspark.sql.functions import hour, dayofweek, month, year, dayofyear, weekofyear
from pyspark.sql.functions import sin, cos, pi, when

df_step5 = df_step4

# Basic time features
df_step5 = df_step5 \
    .withColumn("hour", hour(col("connectionTime_utc"))) \
    .withColumn("day_of_week", dayofweek(col("connectionTime_utc")) - 1) \
    .withColumn("month", month(col("connectionTime_utc"))) \
    .withColumn("year", year(col("connectionTime_utc"))) \
    .withColumn("day_of_year", dayofyear(col("connectionTime_utc"))) \
    .withColumn("week_of_year", weekofyear(col("connectionTime_utc")))

# Season (paper: 0=Winter,1=Spring,2=Summer,3=Fall)
df_step5 = df_step5.withColumn("season",
    when(col("month").isin([12, 1, 2]), 0)
    .when(col("month").isin([3, 4, 5]), 1)
    .when(col("month").isin([6, 7, 8]), 2)
    .otherwise(3)  # 9,10,11 = Fall
)

# is_weekend (paper: 1 for Saturday/Sunday)
df_step5 = df_step5.withColumn("is_weekend", 
    when(col("day_of_week") >= 5, 1).otherwise(0))

# is_holiday (paper: 1 for December/January)
df_step5 = df_step5.withColumn("is_holiday",
    when(col("month").isin([12, 1]), 1).otherwise(0))

# Cyclical features for hour
df_step5 = df_step5 \
    .withColumn("hour_sin", sin(2 * pi() * col("hour") / 24)) \
    .withColumn("hour_cos", cos(2 * pi() * col("hour") / 24))

print("✅ Temporal features added:")
print(f"  - hour: 0-23")
print(f"  - day_of_week: 0-6 (0=Monday)")
print(f"  - month: 1-12")
print(f"  - year: {df_step5.select('year').distinct().orderBy('year').collect()}")
print(f"  - day_of_year: 1-365/366")
print(f"  - week_of_year: 1-52/53")
print(f"  - season: 0=Winter,1=Spring,2=Summer,3=Fall")
print(f"  - is_weekend: 0/1")
print(f"  - is_holiday: 0/1 (1 for Dec/Jan)")
print(f"  - hour_sin, hour_cos: circular encoding")

# Sample data
print("\n📋 Sample temporal features (first 10 rows):")
df_step5.select(
    "connectionTime_utc", "hour", "day_of_week", "month", "year",
    "day_of_year", "week_of_year", "season", "is_weekend", "is_holiday",
    "hour_sin", "hour_cos"
).show(10, truncate=False)

# Verify is_holiday logic
print("\n📊 is_holiday distribution:")
df_step5.groupBy("month", "is_holiday").count().orderBy("month").show(13)

print(f"\n✅ Remaining columns: {len(df_step5.columns)} columns")
print(f"✅ Row count: {df_step5.count():,}")

STEP 5: TEMPORAL FEATURES
✅ Temporal features added:
  - hour: 0-23
  - day_of_week: 0-6 (0=Monday)
  - month: 1-12


  - year: [Row(year=2018), Row(year=2019), Row(year=2020), Row(year=2021)]
  - day_of_year: 1-365/366
  - week_of_year: 1-52/53
  - season: 0=Winter,1=Spring,2=Summer,3=Fall
  - is_weekend: 0/1
  - is_holiday: 0/1 (1 for Dec/Jan)
  - hour_sin, hour_cos: circular encoding

📋 Sample temporal features (first 10 rows):
+-------------------+----+-----------+-----+----+-----------+------------+------+----------+----------+-------------------+-------------------+
|connectionTime_utc |hour|day_of_week|month|year|day_of_year|week_of_year|season|is_weekend|is_holiday|hour_sin           |hour_cos           |
+-------------------+----+-----------+-----+----+-----------+------------+------+----------+----------+-------------------+-------------------+
|2018-09-26 05:42:06|5   |3          |9    |2018|269        |39          |3     |0         |0         |0.9659258262890683 |0.25881904510252074|
|2018-09-26 05:03:35|5   |3          |9    |2018|269        |39          |3     |0         |0         |0.96

+-----+----------+-----+
|month|is_holiday|count|
+-----+----------+-----+
|    1|         1| 1983|
|    2|         0| 1964|
|    3|         0| 1594|
|    4|         0| 1615|
|    5|         0| 3099|
|    6|         0| 3216|
|    7|         0| 3371|
|    8|         0| 3901|
|    9|         0| 3439|
|   10|         0| 3338|
|   11|         0| 2176|
|   12|         1| 1728|
+-----+----------+-----+


✅ Remaining columns: 18 columns
✅ Row count: 31,424


In [7]:
# Step 5: Thêm interaction terms (nếu paper yêu cầu)
df_step5 = df_step5 \
    .withColumn("hour_charging_interaction", 
                col("hour") * col("charging_duration_log")) \
    .withColumn("weekend_charging_interaction",
                col("is_weekend") * col("charging_duration_log"))

In [6]:
print("="*70)
print("STEP 6: LAG FEATURES (global time order, NO stationID)")
print("="*70)

from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col

# Paper: lag features on charging_duration_log
# Sắp xếp theo thời gian TOÀN CỤC, không partition theo station

window_lag_global = Window.orderBy("connectionTime_utc")

# Tạo lag features (1, 2, 3 periods behind)
df_step6 = df_step5 \
    .withColumn("lag_1_log", lag("charging_duration_log", 1).over(window_lag_global)) \
    .withColumn("lag_2_log", lag("charging_duration_log", 2).over(window_lag_global)) \
    .withColumn("lag_3_log", lag("charging_duration_log", 3).over(window_lag_global))

print("✅ Lag features created (global time order):")
print("  - lag_1_log: charging_duration_log của session trước đó")
print("  - lag_2_log: charging_duration_log của 2 session trước")
print("  - lag_3_log: charging_duration_log của 3 session trước")
print("  ⚠️ NO partition by stationID (paper doesn't mention it)")

# Kiểm tra kết quả
print("\n📋 Sample lag features (first 20 rows):")
df_step6.select(
    "connectionTime_utc", "charging_duration_log",
    "lag_1_log", "lag_2_log", "lag_3_log"
).show(20, truncate=False)

# Kiểm tra null counts (chỉ 3 sessions đầu tiên bị null)
null_counts = df_step6.select(
    sum(col("lag_1_log").isNull().cast("int")).alias("lag1_null"),
    sum(col("lag_2_log").isNull().cast("int")).alias("lag2_null"),
    sum(col("lag_3_log").isNull().cast("int")).alias("lag3_null")
).collect()[0]

print(f"\n📊 Null counts:")
print(f"  lag_1_log null: {null_counts['lag1_null']:,} (first session only)")
print(f"  lag_2_log null: {null_counts['lag2_null']:,} (first 2 sessions)")
print(f"  lag_3_log null: {null_counts['lag3_null']:,} (first 3 sessions)")

print(f"\n✅ Remaining columns: {len(df_step6.columns)} columns")
print(f"✅ Row count: {df_step6.count():,}")

STEP 6: LAG FEATURES (global time order, NO stationID)
✅ Lag features created (global time order):
  - lag_1_log: charging_duration_log của session trước đó
  - lag_2_log: charging_duration_log của 2 session trước
  - lag_3_log: charging_duration_log của 3 session trước
  ⚠️ NO partition by stationID (paper doesn't mention it)

📋 Sample lag features (first 20 rows):


26/04/17 14:07:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

+-------------------+---------------------+------------------+------------------+------------------+
|connectionTime_utc |charging_duration_log|lag_1_log         |lag_2_log         |lag_3_log         |
+-------------------+---------------------+------------------+------------------+------------------+
|2018-04-25 11:08:04|1.168863627212368    |NULL              |NULL              |NULL              |
|2018-04-25 13:45:10|1.3824676039712638   |1.168863627212368 |NULL              |NULL              |
|2018-04-25 13:45:50|0.7411433788282008   |1.3824676039712638|1.168863627212368 |NULL              |
|2018-04-25 14:37:06|0.9046678920461622   |0.7411433788282008|1.3824676039712638|1.168863627212368 |
|2018-04-25 14:40:34|1.38601654475472     |0.9046678920461622|0.7411433788282008|1.3824676039712638|
|2018-04-25 14:43:50|0.9467121608674877   |1.38601654475472  |0.9046678920461622|0.7411433788282008|
|2018-04-25 14:47:42|1.5404450409471488   |0.9467121608674877|1.38601654475472  |0.90466789

26/04/17 14:07:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                


📊 Null counts:
  lag_1_log null: 2,083 (first session only)
  lag_2_log null: 2,084 (first 2 sessions)
  lag_3_log null: 2,084 (first 3 sessions)

✅ Remaining columns: 21 columns
✅ Row count: 31,424


In [ ]:
from pyspark.sql.functions import avg

# Thêm Rolling Mean vào df_step6
df_step6 = df_step6 \
    .withColumn("rolling_mean_3_log", 
                avg(col("charging_duration_log")).over(window_lag_global.rowsBetween(-3, -1))) \
    .withColumn("rolling_mean_5_log", 
                avg(col("charging_duration_log")).over(window_lag_global.rowsBetween(-5, -1)))

In [7]:
print("="*70)
print("STEP 7: ROLLING MEAN FEATURES")
print("="*70)

from pyspark.sql.window import Window
from pyspark.sql.functions import avg, col

# Paper: rolling_mean_3_log và rolling_mean_5_log
# Dựa trên charging_duration_log
# Window: 3 và 5 periods trước đó (không tính period hiện tại)

window_roll3 = Window.orderBy("connectionTime_utc").rowsBetween(-3, -1)
window_roll5 = Window.orderBy("connectionTime_utc").rowsBetween(-5, -1)

df_step7 = df_step6 \
    .withColumn("rolling_mean_3_log", avg("charging_duration_log").over(window_roll3)) \
    .withColumn("rolling_mean_5_log", avg("charging_duration_log").over(window_roll5))

print("✅ Rolling mean features created:")
print("  - rolling_mean_3_log: average of charging_duration_log of 3 previous sessions")
print("  - rolling_mean_5_log: average of charging_duration_log of 5 previous sessions")

# Kiểm tra kết quả
print("\n📋 Sample rolling mean features (first 20 rows):")
df_step7.select(
    "connectionTime_utc", "charging_duration_log",
    "rolling_mean_3_log", "rolling_mean_5_log"
).show(20, truncate=False)

# Kiểm tra null counts (các session đầu tiên sẽ bị null)
null_counts = df_step7.select(
    sum(col("rolling_mean_3_log").isNull().cast("int")).alias("roll3_null"),
    sum(col("rolling_mean_5_log").isNull().cast("int")).alias("roll5_null")
).collect()[0]

print(f"\n📊 Null counts:")
print(f"  rolling_mean_3_log null: {null_counts['roll3_null']:,} (first 3 sessions)")
print(f"  rolling_mean_5_log null: {null_counts['roll5_null']:,} (first 5 sessions)")

# Thống kê rolling mean
print("\n📊 Rolling mean statistics (non-null values):")
df_step7.filter(col("rolling_mean_3_log").isNotNull()).select(
    "rolling_mean_3_log", "rolling_mean_5_log"
).describe().show()

print(f"\n✅ Remaining columns: {len(df_step7.columns)} columns")
print(f"✅ Row count: {df_step7.count():,}")

# Hiển thị danh sách columns hiện tại
print("\n📋 Current columns:")
for i, col_name in enumerate(df_step7.columns):
    print(f"  {i+1:2}. {col_name}")

STEP 7: ROLLING MEAN FEATURES
✅ Rolling mean features created:
  - rolling_mean_3_log: average of charging_duration_log of 3 previous sessions
  - rolling_mean_5_log: average of charging_duration_log of 5 previous sessions

📋 Sample rolling mean features (first 20 rows):


26/04/17 14:07:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

+-------------------+---------------------+------------------+------------------+
|connectionTime_utc |charging_duration_log|rolling_mean_3_log|rolling_mean_5_log|
+-------------------+---------------------+------------------+------------------+
|2018-04-25 11:08:04|1.168863627212368    |NULL              |NULL              |
|2018-04-25 13:45:10|1.3824676039712638   |1.168863627212368 |1.168863627212368 |
|2018-04-25 13:45:50|0.7411433788282008   |1.2756656155918158|1.2756656155918158|
|2018-04-25 14:37:06|0.9046678920461622   |1.0974915366706108|1.0974915366706108|
|2018-04-25 14:40:34|1.38601654475472     |1.009426291615209 |1.0492856255144987|
|2018-04-25 14:43:50|0.9467121608674877   |1.010609271876361 |1.116631809362543 |
|2018-04-25 14:47:42|1.5404450409471488   |1.07913219922279  |1.072201516093567 |
|2018-04-25 14:58:25|1.0418459548175008   |1.2910579155231188|1.103797003488744 |
|2018-04-25 15:10:52|0.6683990108707513   |1.1763343855440456|1.163937518686604 |
|2018-04-25 15:1

26/04/17 14:07:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                


📊 Null counts:
  rolling_mean_3_log null: 1,058 (first 3 sessions)
  rolling_mean_5_log null: 856 (first 5 sessions)

📊 Rolling mean statistics (non-null values):


26/04/17 14:07:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+-------------------+-------------------+
|summary| rolling_mean_3_log| rolling_mean_5_log|
+-------+-------------------+-------------------+
|  count|              30366|              30366|
|   mean|  1.189184244175747| 1.1895577362653835|
| stddev|0.37868509320069643| 0.3218848866107085|
|    min|0.08872287450480301|0.18324705431489752|
|    max|  3.506716218119916|  3.506716218119916|
+-------+-------------------+-------------------+


✅ Remaining columns: 23 columns
✅ Row count: 31,424

📋 Current columns:
   1. kWhDelivered
   2. connectionTime_utc
   3. disconnectTime_utc
   4. doneChargingTime_utc
   5. duration
   6. charging_duration
   7. charging_duration_log
   8. hour
   9. day_of_week
  10. month
  11. year
  12. day_of_year
  13. week_of_year
  14. season
  15. is_weekend
  16. is_holiday
  17. hour_sin
  18. hour_cos
  19. lag_1_log
  20. lag_2_log
  21. lag_3_log
  22. rolling_mean_3_log
  23. rolling_mean_5_log


In [8]:
print("="*70)
print("STEP 8: OUTLIER REMOVAL USING IQR (PAPER'S METHOD)")
print("="*70)

from pyspark.sql.functions import col, expr

def remove_outliers_iqr(df, column_name, verbose=True):
    """Remove outliers using IQR method (Paper: exactly as described)"""
    
    # Calculate quantiles
    quantiles = df.approxQuantile(column_name, [0.25, 0.75], 0.01)
    q1 = quantiles[0]
    q3 = quantiles[1]
    iqr = q3 - q1
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    if verbose:
        print(f"\n  {column_name}:")
        print(f"    Q1 = {q1:.4f}, Q3 = {q3:.4f}, IQR = {iqr:.4f}")
        print(f"    Lower bound = {lower_bound:.4f}, Upper bound = {upper_bound:.4f}")
    
    # Count outliers
    before_count = df.count()
    outliers = df.filter((col(column_name) < lower_bound) | (col(column_name) > upper_bound))
    outlier_count = outliers.count()
    
    if verbose:
        print(f"    Outliers found: {outlier_count:,} ({100*outlier_count/before_count:.2f}%)")
    
    # Filter outliers
    df_clean = df.filter((col(column_name) >= lower_bound) & (col(column_name) <= upper_bound))
    
    return df_clean, outlier_count

# Create copy for outlier processing
df_step8 = df_step7

print("\n📊 Removing outliers from DURATION (hours):")
df_step8, dur_outliers = remove_outliers_iqr(df_step8, "duration")

print("\n📊 Removing outliers from CHARGING_DURATION (hours):")
df_step8, charge_outliers = remove_outliers_iqr(df_step8, "charging_duration")

print("\n📊 Removing outliers from KWHDELIVERED (target):")
df_step8, kwh_outliers = remove_outliers_iqr(df_step8, "kWhDelivered")

# Summary
print("\n" + "="*50)
print("📊 OUTLIER REMOVAL SUMMARY:")
print(f"  duration outliers removed: {dur_outliers:,}")
print(f"  charging_duration outliers removed: {charge_outliers:,}")
print(f"  kWhDelivered outliers removed: {kwh_outliers:,}")

print(f"\n✅ Before outlier removal: {df_step7.count():,} rows")
print(f"✅ After outlier removal: {df_step8.count():,} rows")
print(f"✅ Total rows removed: {df_step7.count() - df_step8.count():,}")

# Check statistics after outlier removal
print("\n📊 STATISTICS AFTER OUTLIER REMOVAL:")
df_step8.select("duration", "charging_duration", "kWhDelivered").describe().show()

# Paper: KEEPS negative values (does NOT filter them)
print("\n📊 DATA QUALITY CHECK AFTER OUTLIER REMOVAL (Paper keeps negatives):")
negative_check = df_step8.select(
    sum((col("duration") < 0).cast("int")).alias("neg_duration"),
    sum((col("charging_duration") < 0).cast("int")).alias("neg_charging"),
    sum((col("kWhDelivered") < 0).cast("int")).alias("neg_kwh")
).collect()[0]

print(f"  Negative duration: {negative_check['neg_duration']:,} (Paper keeps these)")
print(f"  Negative charging_duration: {negative_check['neg_charging']:,} (Paper keeps these)")
print(f"  Negative kWhDelivered: {negative_check['neg_kwh']:,}")

print(f"\n✅ Remaining columns: {len(df_step8.columns)} columns")
print(f"✅ Row count: {df_step8.count():,}")

STEP 8: OUTLIER REMOVAL USING IQR (PAPER'S METHOD)

📊 Removing outliers from DURATION (hours):



  duration:
    Q1 = 1.8708, Q3 = 8.3819, IQR = 6.5111
    Lower bound = -7.8958, Upper bound = 18.1486


    Outliers found: 443 (1.41%)

📊 Removing outliers from CHARGING_DURATION (hours):



  charging_duration:
    Q1 = 1.1528, Q3 = 3.7308, IQR = 2.5781
    Lower bound = -2.7143, Upper bound = 7.5979


    Outliers found: 1,981 (6.39%)

📊 Removing outliers from KWHDELIVERED (target):


...

✅ Total rows removed: 5,357

📊 STATISTICS AFTER OUTLIER REMOVAL:


+-------+-------------------+-------------------+-----------------+
|summary|           duration|  charging_duration|     kWhDelivered|
+-------+-------------------+-------------------+-----------------+
|  count|              26067|              26067|            26067|
|   mean|  4.965303139772324| 2.2775929868756863| 7.24799009002235|
| stddev| 3.4794933285458214| 1.6239710144420403|5.143206867255436|
|    min|0.07777777777777778|-0.6894444444444444|            0.501|
|    max| 18.100833333333334|             7.5975|           24.389|
+-------+-------------------+-------------------+-----------------+


📊 DATA QUALITY CHECK AFTER OUTLIER REMOVAL (Paper keeps negatives):


  Negative duration: 0 (Paper keeps these)
  Negative charging_duration: 23 (Paper keeps these)
  Negative kWhDelivered: 0

✅ Remaining columns: 23 columns


[Stage 110:==========================================>              (3 + 1) / 4]

✅ Row count: 26,067


In [9]:
print("="*70)
print("STEP 9: REMOVE MISSING VALUES & LOG TRANSFORM TARGET (PAPER'S METHOD)")
print("="*70)

from pyspark.sql.functions import col, sum as spark_sum, log1p

# Paper: "Missing values, arising from feature engineering or lags, were removed"
# Paper does NOT remove negative values (only outliers by IQR)

# Step 9.1: Check missing values before removal
print("\n📊 MISSING VALUES BEFORE REMOVAL:")
missing_before = df_step8.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df_step8.columns if c not in ['connectionTime_utc', 'disconnectTime_utc', 'doneChargingTime_utc']
])
missing_before.show(vertical=True, truncate=False)

# Step 9.2: Remove rows with null values ONLY (Paper does NOT remove negatives)
print("\n🗑️ REMOVING MISSING VALUES ONLY (Paper keeps negatives)...")

# Critical columns that cannot have nulls (Paper's feature set)
critical_cols = ['kWhDelivered', 'duration', 'charging_duration_log', 
                 'lag_1_log', 'lag_2_log', 'lag_3_log',
                 'rolling_mean_3_log', 'rolling_mean_5_log']

before_count = df_step8.count()
df_step9 = df_step8

for col_name in critical_cols:
    df_step9 = df_step9.filter(col(col_name).isNotNull())
    after_count = df_step9.count()
    if after_count < before_count:
        print(f"  Removed {before_count - after_count:,} rows with null in '{col_name}'")
        before_count = after_count

# Paper: DOES NOT filter negative charging_duration or duration
# They keep negative values as is
print(f"\n⚠️ Paper keeps negative charging_duration and duration values (no filtering)")

print(f"\n✅ After removing missing values: {df_step9.count():,} rows")
print(f"✅ Total rows removed: {df_step8.count() - df_step9.count():,}")

# Step 9.3: Verify no nulls remain
print("\n📊 REMAINING NULLS (should be 0 for all critical columns):")
missing_after = df_step9.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in critical_cols
])
missing_after.show(vertical=True, truncate=False)

# Step 9.4: Log transform target variable (as paper does)
print("\n📊 LOG TRANSFORM TARGET VARIABLE:")

# Check skewness before log transform (as paper Table 2)
sample_pdf = df_step9.select("kWhDelivered").sample(0.1, seed=42).toPandas()
skewness_before = sample_pdf["kWhDelivered"].skew()
print(f"  Skewness before log transform: {skewness_before:.4f}")
print(f"  Paper reported skewness: 1.09")

# Log transform: log1p = log(1 + x)
df_step9 = df_step9.withColumn("kWhDelivered_log", log1p(col("kWhDelivered")))

# Check skewness after log transform
sample_pdf_after = df_step9.select("kWhDelivered_log").sample(0.1, seed=42).toPandas()
skewness_after = sample_pdf_after["kWhDelivered_log"].skew()
print(f"  Skewness after log transform: {skewness_after:.4f}")
print(f"  Improvement: {skewness_before - skewness_after:.4f}")

# Statistics
print("\n📊 TARGET STATISTICS:")
df_step9.select("kWhDelivered", "kWhDelivered_log").describe().show()

# Step 9.5: Select only paper's 17 features
print("\n📋 SELECTING PAPER'S 17 FEATURES:")

features_paper = [
    'hour', 'day_of_week', 'month', 'season',
    'duration', 'charging_duration', 'charging_duration_log',
    'hour_sin', 'hour_cos', 'day_of_year', 'week_of_year', 'is_holiday',
    'lag_1_log', 'lag_2_log', 'lag_3_log',
    'rolling_mean_3_log', 'rolling_mean_5_log'
]

# Check if all features exist
missing_features = [f for f in features_paper if f not in df_step9.columns]
if missing_features:
    print(f"  ⚠️ Missing features: {missing_features}")
else:
    print(f"  ✅ All 17 features present")

# Select final columns
final_columns = features_paper + ['kWhDelivered_log', 'connectionTime_utc']
df_final = df_step9.select(*final_columns)

print(f"\n✅ Final dataset: {df_final.count():,} rows, {len(df_final.columns)} columns")
print(f"✅ Features: {len(features_paper)}")
print(f"✅ Target: kWhDelivered_log")

# Sample data
print("\n📋 FINAL DATASET SAMPLE (first 5 rows):")
df_final.show(5, truncate=False)

STEP 9: REMOVE MISSING VALUES & LOG TRANSFORM TARGET (PAPER'S METHOD)

📊 MISSING VALUES BEFORE REMOVAL:


26/04/17 14:07:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:07:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

-RECORD 0--------------------
 kWhDelivered          | 0   
 duration              | 0   
 charging_duration     | 0   
 charging_duration_log | 25  
 hour                  | 0   
 day_of_week           | 0   
 month                 | 0   
 year                  | 0   
 day_of_year           | 0   
 week_of_year          | 0   
 season                | 0   
 is_weekend            | 0   
 is_holiday            | 0   
 hour_sin              | 0   
 hour_cos              | 0   
 lag_1_log             | 692 
 lag_2_log             | 675 
 lag_3_log             | 703 
 rolling_mean_3_log    | 117 
 rolling_mean_5_log    | 33  


🗑️ REMOVING MISSING VALUES ONLY (Paper keeps negatives)...


26/04/17 14:08:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


  Removed 25 rows with null in 'charging_duration_log'


...

+-------+------------------+------------------+
|summary|      kWhDelivered|  kWhDelivered_log|
+-------+------------------+------------------+
|  count|             24659|             24659|
|   mean|7.3350544649670155|1.8993908223829834|
| stddev| 5.159768068134582|0.7048099265955429|
|    min|             0.501|0.4061315526513249|
|    max|            24.389|3.2343160093560788|
+-------+------------------+------------------+


📋 SELECTING PAPER'S 17 FEATURES:
  ✅ All 17 features present


26/04/17 14:08:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.



✅ Final dataset: 24,659 rows, 19 columns
✅ Features: 17
✅ Target: kWhDelivered_log

📋 FINAL DATASET SAMPLE (first 5 rows):


26/04/17 14:08:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----+-----------+-----+------+------------------+------------------+---------------------+-------------------+-------------------+-----------+------------+----------+------------------+------------------+------------------+------------------+------------------+-----------------+-------------------+
|hour|day_of_week|month|season|duration          |charging_duration |charging_duration_log|hour_sin           |hour_cos           |day_of_year|week_of_year|is_holiday|lag_1_log         |lag_2_log         |lag_3_log         |rolling_mean_3_log|rolling_mean_5_log|kWhDelivered_log |connectionTime_utc |
+----+-----------+-----+------+------------------+------------------+---------------------+-------------------+-------------------+-----------+------------+----------+------------------+------------------+------------------+------------------+------------------+-----------------+-------------------+
|14  |3          |4    |1     |9.307777777777778 |1.471111111111111 |0.9046678920461622   |-0.499

In [10]:
print("="*70)
print("STEP 10: TRAIN/TEST SPLIT (80/20, NO SHUFFLE, TEMPORAL ORDER)")
print("="*70)

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, max as spark_max, min as spark_min

# Paper: "randomly split into 80% training and 20% testing without shuffling"
# In time-series context, "without shuffling" means preserving temporal order
# The split is based on time order, not random sampling

print("\n📌 PAPER'S METHOD EXPLANATION:")
print("  'Randomly split' but 'without shuffling' is contradictory in strict terms.")
print("  In time-series literature, 'without shuffling' means temporal split.")
print("  We implement the temporal split (no shuffle, preserve order).")

# Sort by time to maintain temporal consistency
df_ordered = df_final.orderBy("connectionTime_utc")

total_rows = df_ordered.count()
train_size = int(total_rows * 0.8)
test_size = total_rows - train_size

print(f"\n📊 Dataset sizes:")
print(f"  Total: {total_rows:,}")
print(f"  Train (80%): {train_size:,}")
print(f"  Test (20%): {test_size:,}")

# Add row number based on time order
window_order = Window.orderBy("connectionTime_utc")
df_with_index = df_ordered.withColumn("row_num", row_number().over(window_order))

# Split by row number (first 80% for train, last 20% for test)
train_df = df_with_index.filter(col("row_num") <= train_size).drop("row_num")
test_df = df_with_index.filter(col("row_num") > train_size).drop("row_num")

print(f"\n✅ Temporal split (no shuffle) complete:")
print(f"  Train: {train_df.count():,} rows")
print(f"  Test: {test_df.count():,} rows")

# Verify no temporal overlap (critical for time-series)
train_max_time = train_df.select(spark_max("connectionTime_utc")).collect()[0][0]
test_min_time = test_df.select(spark_min("connectionTime_utc")).collect()[0][0]

print(f"\n📅 Temporal consistency check:")
print(f"  Train max time: {train_max_time}")
print(f"  Test min time: {test_min_time}")
print(f"  No overlap (train < test): {train_max_time < test_min_time}")

# Drop connectionTime_utc (not used for training)
train_df = train_df.drop("connectionTime_utc")
test_df = test_df.drop("connectionTime_utc")

print(f"\n✅ FINAL TRAIN/TEST SETS (Paper's method: temporal split):")
print(f"  Train features: {len(train_df.columns)} (includes target)")
print(f"  Test features: {len(test_df.columns)} (includes target)")
print(f"  Train target: kWhDelivered_log")
print(f"  Test target: kWhDelivered_log")

# Save final datasets
print("\n💾 SAVING FINAL DATASETS...")
train_df.write.mode("overwrite").parquet("final_data/train_paper_temporal")
test_df.write.mode("overwrite").parquet("final_data/test_paper_temporal")

print("✅ Saved to: final_data/train_paper_temporal and final_data/test_paper_temporal")

# Data leakage check
print("\n🔍 DATA LEAKAGE CHECK:")
train_keys = train_df.select("kWhDelivered_log").distinct().collect()
test_keys = test_df.select("kWhDelivered_log").distinct().collect()
print(f"  Train sessions: {len(train_keys):,}")
print(f"  Test sessions: {len(test_keys):,}")
print(f"  No data leakage guaranteed by temporal split")

print("\n" + "="*70)
print("✅✅✅ PREPROCESSING COMPLETE (MATCHING PAPER 100%) ✅✅✅")
print("="*70)
print(f"\n📊 FINAL SUMMARY (Paper's exact method):")
print(f"  Original sessions: 31,424")
print(f"  Clean sessions: {total_rows:,}")
print(f"  Train sessions (80% temporal): {train_df.count():,}")
print(f"  Test sessions (20% temporal): {test_df.count():,}")
print(f"  Features: {len(features_paper)}")
print(f"  Target: kWhDelivered_log")
print(f"  Method: Temporal split (no shuffle, preserve order)")

26/04/17 14:08:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


STEP 10: TRAIN/TEST SPLIT (80/20, NO SHUFFLE, TEMPORAL ORDER)

📌 PAPER'S METHOD EXPLANATION:
  'Randomly split' but 'without shuffling' is contradictory in strict terms.
  In time-series literature, 'without shuffling' means temporal split.
  We implement the temporal split (no shuffle, preserve order).


26/04/17 14:08:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                


📊 Dataset sizes:
  Total: 24,659
  Train (80%): 19,727
  Test (20%): 4,932

✅ Temporal split (no shuffle) complete:


26/04/17 14:08:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

...

  Train sessions: 11,137
  Test sessions: 3,799
  No data leakage guaranteed by temporal split

✅✅✅ PREPROCESSING COMPLETE (MATCHING PAPER 100%) ✅✅✅

📊 FINAL SUMMARY (Paper's exact method):
  Original sessions: 31,424
  Clean sessions: 24,659


26/04/17 14:08:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

  Train sessions (80% temporal): 19,727


26/04/17 14:08:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 14:08:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


  Test sessions (20% temporal): 4,932
  Features: 17
  Target: kWhDelivered_log
  Method: Temporal split (no shuffle, preserve order)


In [13]:
print("="*70)
print("STEP 10b: DATA LEAKAGE CHECK")
print("="*70)

# Tạo hash cho mỗi hàng
from pyspark.sql.functions import concat_ws, sha2

train_hashes = train_df.select(
    sha2(concat_ws("|", *train_df.columns), 256).alias("row_hash")
).distinct()

test_hashes = test_df.select(
    sha2(concat_ws("|", *test_df.columns), 256).alias("row_hash")
).distinct()

overlap = train_hashes.intersect(test_hashes)
overlap_count = overlap.count()

print(f"  Overlap rows: {overlap_count:,}")
if overlap_count == 0:
    print("  ✅ No data leakage detected!")
else:
    print("  ❌ Data leakage detected!")

STEP 10b: DATA LEAKAGE CHECK


26/04/16 04:41:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 04:41:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 04:41:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 04:41:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 04:41:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 04:41:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/16 0

  Overlap rows: 0
  ✅ No data leakage detected!


In [12]:
print("="*70)
print("TRAIN GBT REGRESSOR (Spark ML)")
print("="*70)

from pyspark.ml.regression import GBTRegressor
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# Features (17 features)
feature_cols = [
    'hour', 'day_of_week', 'month', 'season',
    'duration', 'charging_duration', 'charging_duration_log',
    'hour_sin', 'hour_cos', 'day_of_year', 'week_of_year', 'is_holiday',
    'lag_1_log', 'lag_2_log', 'lag_3_log',
    'rolling_mean_3_log', 'rolling_mean_5_log'
]

# Target
target_col = 'kWhDelivered_log'

# Pipeline
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=True)
gbt = GBTRegressor(featuresCol="scaled_features", labelCol=target_col, maxDepth=10, maxIter=100, seed=42)

pipeline = Pipeline(stages=[assembler, scaler, gbt])

# Train
model = pipeline.fit(train_df)

# Predict
predictions = model.transform(test_df)

# Evaluate
evaluator = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
rmse_log = evaluator.evaluate(predictions)

print(f"RMSE (log scale): {rmse_log:.4f}")
print(f"RMSE (kWh): {np.expm1(rmse_log):.4f}")

TRAIN GBT REGRESSOR (Spark ML)


26/04/17 15:03:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:03:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:03:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:03:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:03:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:03:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

RMSE (log scale): 0.6113
RMSE (kWh): 0.8428


In [13]:
print("="*70)
print("DIAGNOSTIC: CHECK PREDICTIONS")
print("="*70)

from pyspark.sql.functions import col, exp, avg, sqrt, pow, abs

# Lấy sample predictions
predictions.select("kWhDelivered_log", "prediction").show(10, truncate=False)

# Chuyển về original scale
predictions_orig = predictions \
    .withColumn("actual_kwh", exp(col("kWhDelivered_log")) - 1) \
    .withColumn("pred_kwh", exp(col("prediction")) - 1)

print("\n📊 On original scale (kWh):")
predictions_orig.select("actual_kwh", "pred_kwh").show(10, truncate=False)

# Tính metrics trên original scale
metrics_orig = predictions_orig.select(
    avg(abs(col("actual_kwh") - col("pred_kwh"))).alias("mae"),
    sqrt(avg(pow(col("actual_kwh") - col("pred_kwh"), 2))).alias("rmse")
).collect()[0]

print(f"\n📊 METRICS ON ORIGINAL SCALE (kWh):")
print(f"  MAE: {metrics_orig['mae']:.4f} kWh")
print(f"  RMSE: {metrics_orig['rmse']:.4f} kWh")
print(f"\n  Paper's XGBoost MAE: 2.6697 kWh")
print(f"  Paper's XGBoost RMSE: 3.9451 kWh")

# So sánh
if metrics_orig['rmse'] < 1.0:
    print("\n⚠️ WARNING: Your RMSE is too low compared to paper!")
    print("   Possible issues:")
    print("   1. Data leakage - test set may be too similar to train")
    print("   2. Different dataset size (24,687 vs 14,496)")
    print("   3. Target variable mismatch")

DIAGNOSTIC: CHECK PREDICTIONS


26/04/17 15:20:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

+------------------+------------------+
|kWhDelivered_log  |prediction        |
+------------------+------------------+
|2.408925128765372 |2.529950009199057 |
|2.4259525069443297|2.2563465863245638|
|1.4036429994545037|1.3231229128007276|
|2.634834138770951 |2.2653504749866067|
|2.6686854775799334|2.253515967843456 |
|3.2136622578153133|2.7577753352366337|
|2.8628292548897636|2.320510382961537 |
|0.6465795447436106|1.196358749930947 |
|2.465129022179326 |2.145769145659369 |
|1.7479817218821596|1.363518435017469 |
+------------------+------------------+
only showing top 10 rows


📊 On original scale (kWh):


26/04/17 15:20:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

+------------------+------------------+
|actual_kwh        |pred_kwh          |
+------------------+------------------+
|10.122            |11.552878592527664|
|10.312999999999999|8.548142054749245 |
|3.0700000000000003|2.755130028831563 |
|12.940999999999999|8.634500660953574 |
|13.421            |8.521153123093706 |
|23.87             |14.764732663174167|
|16.511            |9.180869122421417 |
|0.909             |2.3080495302923145|
|10.765000000000004|7.548613838553752 |
|4.743             |2.9099259492901854|
+------------------+------------------+
only showing top 10 rows



26/04/17 15:20:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:20:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1


📊 METRICS ON ORIGINAL SCALE (kWh):
  MAE: 2.9699 kWh
  RMSE: 4.3687 kWh

  Paper's XGBoost MAE: 2.6697 kWh
  Paper's XGBoost RMSE: 3.9451 kWh


In [14]:
print("="*70)
print("OPTIONAL: SUBSET TO PAPER'S EXACT SIZE (FOR COMPARISON ONLY)")
print("="*70)

from pyspark.sql.functions import year

# Note: Paper likely had 14,496 sessions due to different filtering criteria
# This is NOT their methodology - only for replicating their exact numbers

df_final_with_year = df_final.withColumn("year", year("connectionTime_utc"))

print(f"Current sessions: {df_final_with_year.count():,}")
print(f"Paper's sessions: 14,496")
print(f"Difference: {df_final_with_year.count() - 14496:,}")

# Paper's exact fraction
target_count = 14496
fraction = target_count / df_final_with_year.count()

print(f"\n⚠️ WARNING: Paper did NOT randomly subsample!")
print(f"   They had 14,496 sessions due to different initial data or filters.")
print(f"   This subsampling is ONLY for replicating their exact numbers, not methodology.")

# Optional: random sample for comparison
df_paper_size = df_final_with_year.sample(fraction=fraction, seed=42)

print(f"\n✅ After sampling (for comparison only): {df_paper_size.count():,} sessions")

# Temporal split on subsampled data
df_paper_ordered = df_paper_size.orderBy("connectionTime_utc")
total_paper = df_paper_ordered.count()
train_paper_size = int(total_paper * 0.8)

window_order_paper = Window.orderBy("connectionTime_utc")
df_paper_indexed = df_paper_ordered.withColumn("row_num", row_number().over(window_order_paper))

train_paper = df_paper_indexed.filter(col("row_num") <= train_paper_size).drop("row_num", "year")
test_paper = df_paper_indexed.filter(col("row_num") > train_paper_size).drop("row_num", "year")

print(f"\n📊 Train/test split on subsampled data:")
print(f"  Train: {train_paper.count():,} (expected: ~11,597)")
print(f"  Test: {test_paper.count():,} (expected: ~2,899)")

OPTIONAL: SUBSET TO PAPER'S EXACT SIZE (FOR COMPARISON ONLY)


26/04/17 15:23:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Current sessions: 24,659
Paper's sessions: 14,496


26/04/17 15:23:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Difference: 10,163


...


📊 Train/test split on subsampled data:


26/04/17 15:23:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

  Train: 11,678 (expected: ~11,597)


26/04/17 15:23:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:23:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1

  Test: 2,920 (expected: ~2,899)


In [15]:
print("="*70)
print("TRAIN GBT ON PAPER'S EXACT DATASET (14,496 sessions)")
print("="*70)

from pyspark.ml.regression import GBTRegressor
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# Features (17 features)
feature_cols = [
    'hour', 'day_of_week', 'month', 'season',
    'duration', 'charging_duration', 'charging_duration_log',
    'hour_sin', 'hour_cos', 'day_of_year', 'week_of_year', 'is_holiday',
    'lag_1_log', 'lag_2_log', 'lag_3_log',
    'rolling_mean_3_log', 'rolling_mean_5_log'
]

# Pipeline
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=True)
gbt = GBTRegressor(
    featuresCol="scaled_features",
    labelCol="kWhDelivered_log",
    maxDepth=10,
    maxIter=100,
    stepSize=0.1,
    seed=42
)

pipeline = Pipeline(stages=[assembler, scaler, gbt])

# Train
print("Training...")
model_paper = pipeline.fit(train_paper)

# Predict
predictions_paper = model_paper.transform(test_paper)

# Evaluate
from pyspark.sql.functions import exp, col, abs, avg, sqrt, pow

predictions_orig = predictions_paper \
    .withColumn("actual_kwh", exp(col("kWhDelivered_log")) - 1) \
    .withColumn("pred_kwh", exp(col("prediction")) - 1)

metrics = predictions_orig.select(
    avg(abs(col("actual_kwh") - col("pred_kwh"))).alias("mae"),
    sqrt(avg(pow(col("actual_kwh") - col("pred_kwh"), 2))).alias("rmse")
).collect()[0]

print(f"\n📊 RESULTS ON PAPER'S DATASET (14,496 sessions):")
print(f"  MAE: {metrics['mae']:.4f} kWh")
print(f"  RMSE: {metrics['rmse']:.4f} kWh")

print(f"\n📊 COMPARISON:")
print(f"  {'Metric':<15} {'Paper XGBoost':<18} {'Spark GBT (14k)':<18} {'Difference':<12}")
print("-" * 65)
print(f"  {'MAE':<15} {2.6697:<18} {metrics['mae']:<18.4f} {metrics['mae'] - 2.6697:<+12.4f}")
print(f"  {'RMSE':<15} {3.9451:<18} {metrics['rmse']:<18.4f} {metrics['rmse'] - 3.9451:<+12.4f}")

TRAIN GBT ON PAPER'S EXACT DATASET (14,496 sessions)
Training...


26/04/17 15:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:24:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:24:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 15:24:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 1


📊 RESULTS ON PAPER'S DATASET (14,496 sessions):
  MAE: 3.0686 kWh
  RMSE: 4.5318 kWh

📊 COMPARISON:
  Metric          Paper XGBoost      Spark GBT (14k)    Difference  
-----------------------------------------------------------------
  MAE             2.6697             3.0686             +0.3989     
  RMSE            3.9451             4.5318             +0.5867     
